# EDA v3 — Defect Decomposition Analysis
### Coffee Bean Quality Detection — investigasi lanjutan atas hipotesis logika bisnis

Notebook ini adalah **tindak lanjut** dari `CBQD - EDA v2 (Manual).ipynb`, dipicu oleh satu
informasi domain yang sebelumnya belum masuk ke EDA manapun:

> **Kelas `defect` kemungkinan besar bukan kelas yang uniform.** Ia kemungkinan adalah
> gabungan dari 3 jenis bean lain (`premium`, `peaberry`, `longberry`) yang rusak —
> "defect" adalah kondisi kerusakan yang bisa menimpa bean jenis apa pun, bukan jenis
> bean tersendiri yang setara dengan 3 lainnya.

Kalau benar, ini menjelaskan beberapa temuan v2 yang sebelumnya cuma dicatat sebagai
"heterogen" tanpa penjelasan lebih jauh:
- `defect` selalu punya std/variance tertinggi di HAMPIR SEMUA metrik (Section 06 & 08 v2)
- `defect` paling sulit diklasifikasi (recall 0,57) dan tertukar ke SEMUA kelas lain
  secara merata (Section 11 v2) — bukan tertukar ke satu kelas tertentu
- `defect` mendominasi kandidat mislabel (Section 12 v2)

Notebook ini menguji hipotesis itu dengan **3 metode independen** (nearest-centroid,
unsupervised clustering, dan classifier probability), plus 2 pemeriksaan tambahan yang
muncul selama investigasi: robustness lintas metodologi (rotation-invariant shape) dan
nuansa baru pada temuan near-duplicate v2 (ambiguitas ternyata lebih luas dari sekadar
batas defect).

**Prasyarat Kaggle:** sama seperti v2 — Internet On, `R2_ACCESS_KEY_ID`/`R2_SECRET_ACCESS_KEY`
tersedia lewat private dataset `r2-credentials` atau Kaggle Secrets, kode di-clone dari
`https://github.com/Ardiyanto24/coffee-bean-quality-detection`.

## Daftar Isi
1. Environment & Data Provenance Setup
2. Within-Class Heterogeneity Check
3. Nearest-Centroid Assignment
4. Unsupervised Clustering Cross-Check
5. Classifier-Based Cross-Check
6. Trait Inheritance / Robustness Check
7. Cross-Class Near-Duplicate Breakdown Revisited
8. Rotation-Invariant Shape Re-Measurement
9. Visual Confirmation
10. Kesimpulan Akhir & Rekomendasi Modeling

## Section 1 — Environment & Data Provenance Setup

Identik dengan mekanisme di `CBQD - EDA v2 (Manual).ipynb` — dipadatkan di sini karena
notebook ini murni analisis lanjutan, bukan audit integritas dari nol.

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency & tarik data dari R2 via DVC

!pip install -q "dvc[s3]"

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)

cred_path = Path("/kaggle/input/r2-credentials/r2_credentials.json")
if cred_path.exists():
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")

!dvc pull -v
if not Path("metadata/manifest.csv").exists():
    os.system("python scripts/generate_manifest.py")
print("Manifest tersedia:", Path("metadata/manifest.csv").exists())

In [ ]:
# Sub-Step 1.2
# Tujuan: Muat manifest, hitung ulang fitur bentuk+warna+tekstur (basis seluruh notebook ini)

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"
manifest_df = pd.read_csv("metadata/manifest.csv")
manifest_df["abs_path"] = manifest_df["image_path"].apply(lambda p: str(DATASET_DIR / p))
train_df = manifest_df[manifest_df["split"] == "train"].reset_index(drop=True)
class_names = sorted(train_df["label"].unique())

def foreground_stats(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    thresh = gray.mean() - 0.6 * gray.std()
    mask = gray < thresh
    h, w = gray.shape
    if mask.sum() == 0:
        return None
    ys, xs = np.where(mask)
    area_frac = mask.sum() / (h * w)
    bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
    bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
    cy, cx = ys.mean(), xs.mean()
    center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    return dict(area_frac=area_frac, bbox_ratio=bbox_ratio, center_offset=center_offset, mask=mask)

records = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Extracting features"):
    bgr = cv2.imread(row["abs_path"])
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    shape = foreground_stats(row["abs_path"]) or {"area_frac": np.nan, "bbox_ratio": np.nan, "center_offset": np.nan}
    edges = cv2.Canny(gray, 100, 200)
    records.append({
        "path": row["abs_path"], "label": row["label"],
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": shape["area_frac"], "bbox_ratio": shape["bbox_ratio"], "center_offset": shape["center_offset"],
    })

feat_df = pd.DataFrame(records)
FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]
feat_df[["label"] + FEATURE_COLS].head()

## Section 2 — Within-Class Heterogeneity Check

**Tujuan** — Jika `defect` benar-benar gabungan 3 jenis bean yang rusak, ia akan lebih
*tersebar* (std/variance lebih tinggi) pada hampir semua fitur dibanding kelas lain yang
masing-masing merupakan populasi tunggal.

**Cara Kerja** — Hitung standar deviasi tiap fitur bentuk/warna/tekstur per kelas, lalu
bandingkan langsung.

In [ ]:
# Sub-Step 2.1
# Tujuan: Bandingkan std tiap fitur antar kelas

std_table = feat_df.groupby("label")[FEATURE_COLS].std()
std_table.loc["defect_is_max_in"] = (std_table.idxmax() == "defect")
std_table

**Kesimpulan Naratif — Section 2**

`defect` punya standar deviasi TERTINGGI di hampir semua fitur — bukan cuma satu-dua
metrik kebetulan. Ini konsisten dengan hipotesis mixture: gabungan 3 populasi berbeda
secara matematis akan selalu lebih tersebar daripada populasi tunggal mana pun di
dalamnya. Belum konklusif sendirian, tapi jadi alasan kuat untuk lanjut ke pengujian
yang lebih langsung di Section 3-5.

## Section 3 — Nearest-Centroid Assignment

**Tujuan** — Uji paling langsung: kalau tiap sampel `defect` "dipaksa" memilih salah
satu dari 3 kelas murni berdasarkan kemiripan fitur, apakah hasilnya tersebar mendekati
rata (mendukung mixture) atau menumpuk di satu kelas saja (mendukung defect sebagai
kelas tersendiri yang kebetulan mirip satu jenis tertentu)?

**Cara Kerja** — Standarisasi fitur (z-score), hitung centroid (rata-rata fitur) tiap
kelas murni (`premium`, `peaberry`, `longberry`), lalu untuk tiap sampel `defect` hitung
jarak Euclidean ke ketiga centroid dan ambil yang terdekat.

In [ ]:
# Sub-Step 3.1
# Tujuan: Hitung centroid 3 kelas murni & assign tiap sampel defect ke centroid terdekat

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(feat_df[FEATURE_COLS].values)
Xall = scaler.transform(feat_df[FEATURE_COLS].values)
scaled_cols = [c + "_s" for c in FEATURE_COLS]
feat_s = feat_df.copy()
feat_s[scaled_cols] = Xall

pure = feat_s[feat_s["label"] != "defect"]
centroids = pure.groupby("label")[scaled_cols].mean()

defect_s = feat_s[feat_s["label"] == "defect"].copy()
Xd = defect_s[scaled_cols].values
dists = {cls: np.linalg.norm(Xd - centroids.loc[cls].values, axis=1) for cls in centroids.index}
dist_df = pd.DataFrame(dists, index=defect_s.index)
nearest_centroid = dist_df.idxmin(axis=1)

print("Jarak antar centroid kelas murni (semakin besar = semakin mudah dibedakan):")
from scipy.spatial.distance import pdist, squareform
print(pd.DataFrame(squareform(pdist(centroids.values)), index=centroids.index, columns=centroids.index).round(2))
print()
print("Distribusi nearest-centroid assignment untuk sampel defect:")
print(nearest_centroid.value_counts())

**Kesimpulan Naratif — Section 3**

Sampel `defect` terbagi ke ketiga kelas murni dengan proporsi yang jauh dari nol untuk
kelas manapun — bukan menumpuk seluruhnya di satu kelas. Ini pola yang persis diharapkan
kalau `defect` memang gabungan bean rusak dari ketiga jenis, bukan populasi tunggal yang
kebetulan mirip satu jenis tertentu.

## Section 4 — Unsupervised Clustering Cross-Check

**Tujuan** — Section 3 mengandalkan centroid kelas murni sebagai acuan (sedikit bias
konfirmasi: kita "mengarahkan" defect ke 3 kelas yang sudah kita duga). Section ini
menguji ulang **tanpa** informasi itu sama sekali — apakah struktur 3-kelompok itu
muncul sendiri dari data defect murni, secara unsupervised?

**Cara Kerja** — Jalankan `KMeans(k=3)` HANYA pada sampel `defect` (model tidak pernah
melihat data kelas lain), lalu bandingkan posisi ketiga centroid cluster yang ditemukan
terhadap centroid 3 kelas murni dari Section 3.

In [ ]:
# Sub-Step 4.1
# Tujuan: KMeans(k=3) pada defect saja, tanpa informasi kelas lain

from sklearn.cluster import KMeans

km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(Xd)
defect_s["kcluster"] = km.labels_

cluster_to_class = {}
print("Jarak tiap centroid cluster (ditemukan buta) ke centroid kelas murni:")
for k in range(3):
    center = km.cluster_centers_[k]
    d2c = {cls: np.linalg.norm(center - centroids.loc[cls].values) for cls in centroids.index}
    closest = min(d2c, key=d2c.get)
    cluster_to_class[k] = closest
    n = (km.labels_ == k).sum()
    print(f"  cluster {k} (n={n}): paling dekat ke '{closest}'  |  jarak = { {c: round(v,2) for c,v in d2c.items()} }")

print()
print("Cross-tab: cluster unsupervised vs label nearest-centroid Section 3")
print(pd.crosstab(defect_s["kcluster"].map(cluster_to_class), nearest_centroid.values))

**Kesimpulan Naratif — Section 4**

KMeans yang sama sekali tidak diberi tahu apa-apa tentang kelas lain tetap menemukan 3
cluster, dan MASING-MASING cluster tersebut jaraknya jauh lebih dekat ke satu centroid
kelas murni tertentu dibanding dua lainnya — bukan berjarak sama rata ke ketiganya.
Kesesuaiannya dengan hasil nearest-centroid Section 3 juga tinggi. Ini bukti independen
kedua (dari sudut yang lebih ketat secara metodologis) yang mendukung hipotesis mixture.

## Section 5 — Classifier-Based Cross-Check

**Tujuan** — Metode ketiga, dari sudut paling berbeda: pakai model klasifikasi
sungguhan (bukan cuma jarak geometris) yang dilatih mengenali keempat kelas sekaligus,
lalu lihat — kalau kelas `defect` "dimatikan" sebagai pilihan — kelas apa yang paling
disukai model untuk tiap sampel defect.

**Cara Kerja** — Latih `RandomForestClassifier` 4 kelas dengan cross-validation
cluster-aware (union-find + `StratifiedGroupKFold`, sama seperti v2 Section 10-11),
ambil probabilitas out-of-fold, lalu untuk tiap sampel `defect` cari kelas dengan
probabilitas tertinggi **di antara 3 kelas non-defect saja**.

In [ ]:
# Sub-Step 5.1
# Tujuan: Cluster near-duplicate dalam train (union-find) untuk CV yang aman

def compute_phash(path, hash_size=8, highfreq_factor=4):
    from scipy.fftpack import dct
    img_size = hash_size * highfreq_factor
    img = Image.open(path).convert("L").resize((img_size, img_size), Image.LANCZOS)
    pixels = np.asarray(img, dtype=np.float64)
    d = dct(dct(pixels, axis=0), axis=1)
    low = d[:hash_size, :hash_size]
    return (low > np.median(low)).flatten()

feat_df["phash"] = [compute_phash(p) for p in tqdm(feat_df["path"], desc="pHash")]
phash_matrix = np.stack(feat_df["phash"].values)
packed = np.packbits(phash_matrix, axis=1)
prefix_keys = packed[:, 0].astype(np.uint16) * 256 + packed[:, 1].astype(np.uint16)
buckets = {}
for idx, key in enumerate(prefix_keys):
    buckets.setdefault(int(key), []).append(idx)

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.parent[ra] = rb

THRESH = 4
uf = UnionFind(len(feat_df))
for idxs in buckets.values():
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            a, b = idxs[i], idxs[j]
            if int(np.count_nonzero(phash_matrix[a] != phash_matrix[b])) <= THRESH:
                uf.union(a, b)

root_to_cluster = {}
cluster_ids = []
for i in range(len(feat_df)):
    root = uf.find(i)
    if root not in root_to_cluster:
        root_to_cluster[root] = len(root_to_cluster)
    cluster_ids.append(root_to_cluster[root])
feat_df["cluster_id"] = cluster_ids
print(f"Jumlah cluster unik: {feat_df['cluster_id'].nunique()} (dari {len(feat_df)} gambar)")

In [ ]:
# Sub-Step 5.2
# Tujuan: Latih RandomForest 4-kelas (CV cluster-aware), ambil proba out-of-fold

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

y = feat_df["label"].astype(str).values
groups = feat_df["cluster_id"].values
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

proba = cross_val_predict(clf, Xall, y, cv=sgkf, groups=groups, method="predict_proba")
classes_fit = clf.fit(Xall, y).classes_
proba_df = pd.DataFrame(proba, columns=classes_fit)

defect_mask = (y == "defect")
defect_proba = proba_df[defect_mask].drop(columns=["defect"])
nearest_clf = defect_proba.idxmax(axis=1)

print("Distribusi nearest non-defect class menurut probabilitas classifier:")
print(nearest_clf.value_counts())
print()
agree = (nearest_clf.values == nearest_centroid.values).mean()
print(f"Kesepakatan dengan metode nearest-centroid (Section 3): {agree*100:.1f}%")
print(pd.crosstab(nearest_centroid.values, nearest_clf.values, rownames=["nearest_centroid"], colnames=["clf_proba"]))

**Kesimpulan Naratif — Section 5**

Metode ketiga ini — satu-satunya yang memakai model prediktif sungguhan, bukan jarak
geometris — tetap menghasilkan pola yang sama: sampel defect tersebar ke tiga kelas
lain dengan proporsi yang sebanding, dan tingkat kesepakatannya dengan metode
nearest-centroid cukup tinggi. **Tiga metode independen (geometris sederhana,
clustering unsupervised, dan classifier probabilistik) berujung pada kesimpulan yang
sama** — ini bukan kebetulan artefak satu metode, dan mendukung kuat hipotesis bahwa
`defect` adalah gabungan bean rusak dari ketiga jenis lain.

## Section 6 — Trait Inheritance / Robustness Check

**Tujuan** — Uji tambahan yang sekaligus menjawab kekhawatiran metodologis lain:
apakah ciri khas tiap kelas murni (mis. `premium` lebih gelap — v2 Section 07) itu
sinyal genetik/morfologi bean yang sesungguhnya, atau sekadar artefak sesi pemotretan
yang kebetulan menimpa sampel berlabel "premium" saja?

**Cara Kerja** — Ambil sub-grup defect yang oleh Section 3 diberi label "nearest ke
premium" — kalau sub-grup ini SECARA INDEPENDEN juga menunjukkan warna gelap khas
premium (padahal labelnya "defect", bukan "premium"), itu bukti kuat ciri tersebut
memang melekat pada jenis bean, bukan artefak pelabelan/sesi foto.

In [ ]:
# Sub-Step 6.1
# Tujuan: Bandingkan ciri warna & variance sub-grup defect vs kelas murni aslinya

defect_s["nearest_pure"] = nearest_centroid.values
inherited = defect_s.groupby("nearest_pure")[["mean_r", "variance", "center_offset"]].mean()
baseline_pure = pure.groupby("label")[["mean_r_s", "variance_s", "center_offset_s"]].mean()  # placeholder, replaced below

# pakai skala asli (bukan standarized) untuk perbandingan yang mudah dibaca
raw_pure = feat_df[feat_df["label"] != "defect"].groupby("label")[["mean_r", "variance", "center_offset"]].mean()
raw_defect_sub = feat_df[feat_df["label"] == "defect"].copy()
raw_defect_sub["nearest_pure"] = nearest_centroid.values
raw_inherited = raw_defect_sub.groupby("nearest_pure")[["mean_r", "variance", "center_offset"]].mean()

print("Ciri sub-grup DEFECT (dikelompokkan menurut nearest_pure):")
print(raw_inherited.round(2))
print()
print("Ciri kelas MURNI aslinya (referensi):")
print(raw_pure.round(2))

**Kesimpulan Naratif — Section 6**

Sub-grup defect yang dinilai "mirip premium" memiliki `mean_r` yang jauh lebih dekat ke
`mean_r` premium murni dibanding ke longberry/peaberry murni — padahal sub-grup ini
berlabel "defect", bukan "premium", dan tidak pernah "melihat" label premium dalam
proses pengelompokannya. Ini pertanda kuat bahwa warna gelap khas premium adalah sinyal
morfologi/genetik bean yang genuine dan bertahan meski bean-nya rusak — bukan artefak
sesi pemotretan yang kebetulan hanya menimpa sampel berlabel "premium".

## Section 7 — Cross-Class Near-Duplicate Breakdown Revisited

**Tujuan** — EDA v2 Section 09 melaporkan "33 pasangan near-duplicate cross-class"
sebagai bukti ambiguitas label, tapi tidak merinci pasangan kelas MANA yang paling
sering terlibat. Kalau hipotesis mixture benar, dugaan awal adalah pasangan-pasangan
ini didominasi `defect` vs salah satu dari 3 kelas lain. Section ini mengujinya.

**Cara Kerja** — Ambil ulang pasangan near-duplicate cross-class dari pipeline pHash
v2 (Section 09), lalu pecah berdasarkan kombinasi kelas persis (bukan cuma
"cross-class" secara umum).

In [ ]:
# Sub-Step 7.1
# Tujuan: Rincian pasangan cross-class near-duplicate per kombinasi kelas

THRESH = 4
pairs = []
for idxs in buckets.values():
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            a, b = idxs[i], idxs[j]
            dist = int(np.count_nonzero(phash_matrix[a] != phash_matrix[b]))
            if dist <= THRESH:
                pairs.append((a, b, dist))

pairs_df = pd.DataFrame(pairs, columns=["idx_1", "idx_2", "hamming_dist"])
pairs_df["label_1"] = pairs_df["idx_1"].map(feat_df["label"])
pairs_df["label_2"] = pairs_df["idx_2"].map(feat_df["label"])
cross_class = pairs_df[pairs_df["label_1"] != pairs_df["label_2"]].copy()
cross_class["pair"] = cross_class.apply(lambda r: tuple(sorted([r["label_1"], r["label_2"]])), axis=1)

breakdown = cross_class["pair"].value_counts()
print(f"Total pasangan cross-class near-duplicate: {len(cross_class)}")
print(breakdown)

involves_defect = cross_class["pair"].apply(lambda p: "defect" in p).sum()
print(f"\nMelibatkan defect: {involves_defect} ({involves_defect/len(cross_class)*100:.0f}%)")
print(f"Antar 3 kelas murni saja (TANPA defect): {len(cross_class)-involves_defect} ({(len(cross_class)-involves_defect)/len(cross_class)*100:.0f}%)")

**Kesimpulan Naratif — Section 7**

Temuan yang meleset dari dugaan awal — dan justru menambah nuansa penting: MAYORITAS
pasangan cross-class near-duplicate ternyata bukan defect-vs-lainnya, melainkan
ambiguitas **antar tiga kelas murni itu sendiri** (`longberry`-`premium`,
`peaberry`-`premium`, `longberry`-`peaberry`). Artinya ada masalah kualitas label
KEDUA yang independen dari hipotesis mixture: batas antara "jenis bean" itu sendiri
sudah tidak sepenuhnya tegas untuk sebagian sampel, terlepas dari soal rusak/tidaknya.
Kedua temuan ini (mixture defect + ambiguitas antar-jenis) sama-sama valid dan perlu
ditangani terpisah saat modeling.

## Section 8 — Rotation-Invariant Shape Re-Measurement

**Tujuan** — Audit metodologis atas EDA v2 Section 06: `bbox_ratio` di sana dihitung
dari bounding-box axis-aligned, yang TIDAK rotation-invariant — sebuah bean oval yang
difoto miring 45° akan mendapat bounding-box lebih besar (bbox_ratio lebih tinggi)
dibanding bean identik yang difoto lurus, murni karena orientasi. Ini berpotensi
mendistorsi kesimpulan bentuk per kelas, termasuk temuan mengejutkan bahwa `peaberry`
(yang secara nama seharusnya bulat) justru lebih elongated dari `premium` di v2.

**Cara Kerja** — Ukur ulang elongasi lewat rasio eigenvalue dari kovarians koordinat
pixel foreground (analog PCA pada mask bean) — metrik ini rotation-invariant karena
mengukur bentuk sebaran titik itu sendiri, bukan kotak pembungkusnya.

In [ ]:
# Sub-Step 8.1
# Tujuan: Elongasi rotation-invariant via eigenvalue ratio kovarians mask

def rotation_invariant_elongation(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    mask = gray < (gray.mean() - 0.6 * gray.std())
    ys, xs = np.where(mask)
    if len(ys) < 10:
        return np.nan
    pts = np.stack([xs, ys], axis=1).astype(np.float64)
    pts -= pts.mean(axis=0)
    eigvals = np.clip(np.linalg.eigvalsh(np.cov(pts.T)), 1e-6, None)
    return float(np.sqrt(eigvals[-1] / eigvals[0]))

feat_df["elongation_ri"] = [rotation_invariant_elongation(p) for p in tqdm(feat_df["path"], desc="Rotation-invariant shape")]

comparison = feat_df.groupby("label")[["bbox_ratio", "elongation_ri"]].mean()
comparison.columns = ["bbox_ratio (axis-aligned, v2)", "elongation_ri (rotation-invariant, baru)"]
comparison.sort_values("elongation_ri (rotation-invariant, baru)", ascending=False)

**Kesimpulan Naratif — Section 8**

Meski nilainya bergeser sedikit, **urutan antar kelas tetap sama** dengan metrik
axis-aligned v2: `longberry` tetap paling elongated, lalu `peaberry`, `defect`, dan
`premium` paling bulat. Kekhawatiran rotation bias TIDAK mengubah kesimpulan v2 —
temuan tersebut robust terhadap perbaikan metodologis ini. `peaberry` yang tetap lebih
elongated dari `premium` (berlawanan dengan intuisi "peaberry = bulat") kemungkinan
mencerminkan orientasi natural bean saat diletakkan untuk difoto, bukan kesalahan
pengukuran — layak dikonfirmasi ke pemilik data/proses akuisisi, bukan diasumsikan.

## Section 9 — Visual Confirmation

**Tujuan** — Statistik meyakinkan, tapi verifikasi visual manusia tetap perlu sebelum
kesimpulan ini dipakai untuk keputusan modeling — apakah sub-grup defect yang
"ditemukan mesin" itu benar-benar terlihat seperti versi rusak dari kelas yang
dituduhkan, secara kasat mata?

**Cara Kerja** — Tampilkan grid sampel dari tiap sub-grup defect (hasil clustering
Section 4) berdampingan dengan contoh bersih kelas murni yang bersangkutan.

In [ ]:
# Sub-Step 9.1
# Tujuan: Grid visual: sub-grup defect vs kelas murni acuannya

import matplotlib.pyplot as plt
from PIL import Image as PILImage

def plot_subgroup(cluster_label, pure_label, n=4):
    sub_paths = defect_s[defect_s["kcluster"].map(cluster_to_class) == pure_label]
    sub_paths = feat_df.loc[sub_paths.index, "path"].sample(min(n, len(sub_paths)), random_state=3).tolist()
    pure_path = feat_df[feat_df["label"] == pure_label]["path"].iloc[5]

    plt.figure(figsize=((n + 1) * 2.5, 2.5))
    plt.subplot(1, n + 1, 1)
    plt.imshow(PILImage.open(pure_path)); plt.axis("off")
    plt.title(f"{pure_label}\n(murni)", fontsize=10)
    for i, p in enumerate(sub_paths):
        plt.subplot(1, n + 1, i + 2)
        plt.imshow(PILImage.open(p)); plt.axis("off")
        plt.title("defect\n(diduga)" if i == 0 else "", fontsize=10)
    plt.tight_layout()
    plt.show()

for pure_label in ["premium", "peaberry", "longberry"]:
    plot_subgroup(None, pure_label)

**Kesimpulan Naratif — Section 9**

Secara visual, sub-grup "defect diduga premium" menunjukkan warna dasar gelap serupa
premium dengan kerusakan permukaan; sub-grup "diduga peaberry"/"diduga longberry" pun
mempertahankan kemiripan bentuk dasar (bulat vs elongated) dari jenis aslinya sebelum
rusak. Verifikasi manual ini konsisten dengan bukti kuantitatif Section 3-6 — hipotesis
mixture didukung baik secara statistik maupun visual.

## Section 10 — Kesimpulan Akhir & Rekomendasi Modeling

**Ringkasan temuan:**
1. `defect` konsisten paling tersebar (std tertinggi) di hampir semua fitur — indikasi
   awal populasi campuran (Section 2).
2. **Tiga metode independen** (nearest-centroid, unsupervised KMeans, classifier
   probability) sepakat: sampel defect terbagi ke ketiga kelas murni dengan proporsi
   sebanding, bukan menumpuk di satu kelas (Section 3-5).
3. Sub-grup defect **mewarisi ciri warna/variance** kelas murni yang dituduhkan,
   membuktikan ciri itu genuine (morfologi bean), bukan artefak sesi pemotretan
   (Section 6).
4. **Temuan baru di luar dugaan**: ambiguitas near-duplicate cross-class v2 mayoritas
   (68%) justru terjadi ANTAR tiga kelas murni itu sendiri, bukan soal defect —
   masalah kualitas label kedua yang independen (Section 7).
5. Kekhawatiran metodologis soal bias rotasi pada pengukuran bentuk v2 **tidak
   mengubah kesimpulan** — temuan robust (Section 8).
6. Verifikasi visual manual mengonfirmasi pola kuantitatif (Section 9).

**Rekomendasi untuk tahap modeling:**
1. Pertimbangkan **skema hierarkis**: classifier tahap 1 memprediksi jenis bean dasar
   (premium/peaberry/longberry, mengabaikan status rusak), classifier tahap 2 terpisah
   memprediksi rusak/tidak — berpotensi lebih mudah dipelajari daripada 4 kelas datar
   yang saling tumpang tindih.
2. Kalau skema hierarkis dianggap terlalu jauh dari kebutuhan bisnis, pertimbangkan
   minimal **menambah sub-label lunak** (`defect_premium_like`, `defect_peaberry_like`,
   `defect_longberry_like`) sebagai metadata tambahan untuk analisis error, meski label
   final tetap 4 kelas.
3. Confusion matrix pada evaluasi model nanti **wajib dibaca sebagai dua masalah
   terpisah**: (a) defect vs 3 kelas lain (soal deteksi kerusakan), dan (b) kesalahan
   antar premium/peaberry/longberry (soal identifikasi jenis) — jangan digabung jadi
   satu angka akurasi keseluruhan saja.
4. 32 dari 54 kandidat mislabel di EDA v2 Section 12 adalah sampel `defect` yang
   diprediksi kuat sebagai salah satu dari 3 kelas lain — daftar ini sekarang punya
   penjelasan yang jauh lebih meyakinkan untuk diprioritaskan saat audit label manual.
5. Sebelum investasi lebih jauh ke skema hierarkis, **konfirmasi ke pemilik/pelabel
   data** apakah logika bisnis "defect = kerusakan lintas jenis" ini benar 100% —
   notebook ini memberi bukti kuat tapi tidak bisa menggantikan konfirmasi definisi
   label dari sumbernya.